In [ ]:
# Source - https://stackoverflow.com/a/63519730
# Posted by G M, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-02, License - CC BY-SA 4.0

if 'google.colab' in str(get_ipython()):
  COLAB=True
  print('Running on CoLab')
else:
  COLAB=False
  print('Not running on CoLab')

In [ ]:
if COLAB:
    !git clone https://github.com/avasenin-14/advanced-deep-learning.git
    %cd advanced-deep-learning

In [ ]:
!pip install -r requirements.txt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("machowe/warehouse-object-detection-dataset", output_dir="data/warehouse")

print("Path to dataset files:", path)

In [ ]:
from torchvision.models import swin_v2_t
pretrained_model = swin_v2_t()

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
from pathlib import Path
import json

# Resolve dataset root whether kagglehub returns .../warehouse or .../warehouse_detection_dataset
dataset_root = Path(path)
if (dataset_root / "warehouse_detection_dataset").exists():
    dataset_root = dataset_root / "warehouse_detection_dataset"

raw_root = dataset_root / "raw"
raw_ann_root = raw_root / "train" / "annotations"
raw_img_root = raw_root / "train" / "images"

print("Dataset root:", dataset_root)
print("RAW root:", raw_root)
print("RAW exists:", raw_root.exists())

In [ ]:
# RAW format sanity check: parse one random sample and draw/print annotations.
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

split = "train"
raw_img_dir = dataset_root / "raw" / split / "images"
raw_ann_dir = dataset_root / "raw" / split / "annotations"

with open(dataset_root / "class_mapping.json", "r", encoding="utf-8") as file:
    class_mapping = json.load(file)
id_to_name = {int(item["id"]): item["name"] for item in class_mapping["classes"]}
name_to_id = {item["name"]: int(item["id"]) for item in class_mapping["classes"]}

# Dataset contains a typo in some raw labels.
name_aliases = {"barel": "barrel"}

sample_img_path = random.choice(sorted([p for p in raw_img_dir.glob("*") if p.is_file()]))
stem = sample_img_path.stem
bbox_path = raw_ann_dir / f"{stem}_bbox.npy"
labels_path = raw_ann_dir / f"{stem}_labels.json"

img = Image.open(sample_img_path).convert("RGB")
img_w, img_h = img.size

bboxes = np.load(bbox_path, allow_pickle=True)
with open(labels_path, "r", encoding="utf-8") as file:
    raw_labels = json.load(file)

print(f"[RAW] Image: {sample_img_path.name} | size=({img_w}x{img_h})")
print(f"[RAW] BBox file: {bbox_path.name} | shape={getattr(bboxes, 'shape', None)} | dtype={bboxes.dtype}")
if getattr(bboxes.dtype, "names", None):
    print(f"[RAW] BBox fields: {bboxes.dtype.names}")
print(f"[RAW] Labels file: {labels_path.name} | labels_keys={len(raw_labels)}")


def clean_class_name(raw_class_value):
    tokens = [token.strip() for token in str(raw_class_value).split(",") if token.strip()]
    filtered = [token for token in tokens if token != "data"]
    if not filtered:
        return "unknown"
    name = filtered[0]
    return name_aliases.get(name, name)


def parse_bbox_row(row, idx):
    semantic_key = str(idx)

    if hasattr(row, "dtype") and row.dtype.names:
        fields = set(row.dtype.names)
        if "semanticId" in fields:
            semantic_key = str(int(row["semanticId"]))

        if {"x_min", "y_min", "x_max", "y_max"}.issubset(fields):
            x1 = float(row["x_min"])
            y1 = float(row["y_min"])
            x2 = float(row["x_max"])
            y2 = float(row["y_max"])
            x, y, w, h = x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)
            return semantic_key, x, y, w, h

    values = np.asarray(row).reshape(-1).astype(float)

    if len(values) >= 5:
        semantic_key = str(int(values[0]))
        x1, y1, x2, y2 = values[1], values[2], values[3], values[4]
        x, y, w, h = x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)
        return semantic_key, x, y, w, h

    if len(values) >= 4:
        x1, y1, x2, y2 = values[0], values[1], values[2], values[3]
        x, y, w, h = x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)
        return semantic_key, x, y, w, h

    return semantic_key, 0.0, 0.0, 0.0, 0.0


parsed_annotations = []
for idx, row in enumerate(bboxes):
    semantic_key, x, y, w, h = parse_bbox_row(row, idx)
    label_info = raw_labels.get(semantic_key, {})
    class_name = clean_class_name(label_info.get("class", "unknown"))
    class_id = name_to_id.get(class_name, -1)

    parsed_annotations.append({
        "semantic_key": semantic_key,
        "class_id": class_id,
        "class_name": class_name,
        "bbox": [x, y, w, h],
    })

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img)

for ann in parsed_annotations:
    x, y, w, h = ann["bbox"]
    class_id = ann["class_id"]

    rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor="yellow", facecolor="none")
    ax.add_patch(rect)

    label_text = str(class_id) if class_id >= 0 else "?"
    ax.text(
        x + 2,
        y + 12 if y + 12 <= img_h else y - 2,
        label_text,
        color="black",
        fontsize=8,
        fontweight="bold",
        bbox={"facecolor": "yellow", "alpha": 0.8, "pad": 1, "edgecolor": "none"},
    )

print(f"[RAW] Parsed annotations: {len(parsed_annotations)}")
preview_count = min(20, len(parsed_annotations))
for idx, ann in enumerate(parsed_annotations[:preview_count], start=1):
    x, y, w, h = ann["bbox"]
    print(
        f"{idx:02d}. semantic_key={ann['semantic_key']} | class_id={ann['class_id']} ({ann['class_name']}) | "
        f"bbox=[x={x:.1f}, y={y:.1f}, w={w:.1f}, h={h:.1f}]"
    )
if len(parsed_annotations) > preview_count:
    print(f"... truncated {len(parsed_annotations) - preview_count} additional annotations")

ax.set_title(f"RAW sample: {sample_img_path.name} | boxes: {len(parsed_annotations)}")
ax.axis("off")
plt.show()

In [ ]:
# RAW -> multi-label classification dataset
from pathlib import Path
import json

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import torch


with open(dataset_root / "class_mapping.json", "r", encoding="utf-8") as file:
    class_mapping = json.load(file)

class_names = [item["name"] for item in sorted(class_mapping["classes"], key=lambda x: x["id"])]
name_to_id = {name: idx for idx, name in enumerate(class_names)}

# Background classes dominate this dataset; exclude them by default for better classification signal.
background_names = {"wall", "ceiling", "floor", "pillar"}
name_aliases = {"barel": "barrel"}


def parse_raw_class_list(labels_json_path, include_background=False):
    with open(labels_json_path, "r", encoding="utf-8") as file:
        raw_labels = json.load(file)

    labels = set()
    for _, item in raw_labels.items():
        raw_value = item.get("class", "")
        tokens = [token.strip() for token in str(raw_value).split(",") if token.strip()]
        for token in tokens:
            if token == "data":
                continue
            token = name_aliases.get(token, token)
            if not include_background and token in background_names:
                continue
            if token in name_to_id:
                labels.add(name_to_id[token])
    return sorted(labels)


class RawMultiLabelDataset(Dataset):
    def __init__(
        self,
        dataset_root,
        split="train",
        image_size=224,
        include_background=False,
        use_cache=True,
        refresh_cache=False,
    ):
        self.dataset_root = Path(dataset_root)
        self.split = split
        self.include_background = include_background
        self.use_cache = use_cache

        self.img_dir = self.dataset_root / "raw" / split / "images"
        self.ann_dir = self.dataset_root / "raw" / split / "annotations"

        self.cache_used = False
        cache_dir = self.dataset_root / "raw" / "_cache"
        cache_dir.mkdir(parents=True, exist_ok=True)
        self.cache_path = cache_dir / f"multilabel_index_{split}_bg{int(include_background)}.json"

        self.samples = []

        if self.use_cache and self.cache_path.exists() and not refresh_cache:
            with open(self.cache_path, "r", encoding="utf-8") as file:
                payload = json.load(file)

            for record in payload.get("samples", []):
                img_path = self.img_dir / record["file_name"]
                class_ids = record["class_ids"]
                if img_path.exists() and class_ids:
                    self.samples.append((img_path, class_ids))

            self.cache_used = len(self.samples) > 0

        if not self.samples:
            image_paths = sorted([p for p in self.img_dir.glob("*.png") if p.is_file()])

            for img_path in image_paths:
                labels_path = self.ann_dir / f"{img_path.stem}_labels.json"
                if not labels_path.exists():
                    continue

                class_ids = parse_raw_class_list(
                    labels_path,
                    include_background=self.include_background,
                )
                if not class_ids:
                    continue

                self.samples.append((img_path, class_ids))

            if self.use_cache:
                payload = {
                    "split": split,
                    "include_background": include_background,
                    "samples": [
                        {"file_name": img_path.name, "class_ids": class_ids}
                        for img_path, class_ids in self.samples
                    ],
                }
                with open(self.cache_path, "w", encoding="utf-8") as file:
                    json.dump(payload, file)

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, class_ids = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        target = torch.zeros(len(class_names), dtype=torch.float32)
        for class_id in class_ids:
            target[class_id] = 1.0

        return image, target, img_path.name

In [ ]:
# Build loaders with timing-friendly defaults for notebook debugging.
import time
from torch.utils.data import DataLoader

batch_size = 16
image_size = 224
include_background = False
build_val_test = True
refresh_cache = False

# Runtime policy: use GPU only when running in Colab with CUDA; otherwise force CPU.
use_colab_gpu = bool(COLAB and torch.cuda.is_available())
runtime_device = "cuda" if use_colab_gpu else "cpu"
num_workers = 2 if use_colab_gpu else 0
use_pin_memory = use_colab_gpu

print(f"Runtime device policy -> {runtime_device} (COLAB={COLAB}, cuda_available={torch.cuda.is_available()})")
if use_colab_gpu:
    print("CUDA device:", torch.cuda.get_device_name(0))

start = time.perf_counter()
train_ds = RawMultiLabelDataset(
    dataset_root,
    split="train",
    image_size=image_size,
    include_background=include_background,
    use_cache=True,
    refresh_cache=refresh_cache,
)
train_ds_time = time.perf_counter() - start

if build_val_test:
    t_val = time.perf_counter()
    val_ds = RawMultiLabelDataset(
        dataset_root,
        split="val",
        image_size=image_size,
        include_background=include_background,
        use_cache=True,
        refresh_cache=refresh_cache,
    )
    val_ds_time = time.perf_counter() - t_val

    t_test = time.perf_counter()
    test_ds = RawMultiLabelDataset(
        dataset_root,
        split="test",
        image_size=image_size,
        include_background=include_background,
        use_cache=True,
        refresh_cache=refresh_cache,
    )
    test_ds_time = time.perf_counter() - t_test
else:
    val_ds = None
    test_ds = None
    val_ds_time = 0.0
    test_ds_time = 0.0

print("Dataset sizes (non-empty labels):")
print("train:", len(train_ds), "val:", len(val_ds) if val_ds else "skipped", "test:", len(test_ds) if test_ds else "skipped")
print(f"train init: {train_ds_time:.2f}s | cache used: {train_ds.cache_used}")
if build_val_test:
    print(f"val init: {val_ds_time:.2f}s | cache used: {val_ds.cache_used}")
    print(f"test init: {test_ds_time:.2f}s | cache used: {test_ds.cache_used}")

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
)

if build_val_test:
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=use_pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=use_pin_memory)

batch_start = time.perf_counter()
images, targets, names = next(iter(train_loader))
batch_time = time.perf_counter() - batch_start

print(f"First batch load: {batch_time:.2f}s")
print("Batch shapes:", images.shape, targets.shape)
print("First image:", names[0])
print("First image labels:", [class_names[i] for i in torch.where(targets[0] > 0)[0].tolist()])

In [ ]:
# Train ResNet18 on RAW labels and compare with existing baselines.
import torch
import torch.nn as nn
from torchvision.models import resnet18

# Pull baseline values if they exist from earlier runs in this kernel.
swin_prev_val_f1 = globals().get("swin_best_val_f1", None)
vit_prev_val_f1 = globals().get("vit_best_val_f1", None)

# Respect runtime policy from the loader cell: Colab+CUDA => GPU, otherwise CPU.
device = globals().get("runtime_device", "cpu")
if device not in ["cpu", "cuda"]:
    device = "cpu"

# Auto-limit work on CPU for practical runtime.
epochs = 3
max_train_batches = None if device == "cuda" else 120
max_val_batches = None if device == "cuda" else 40
threshold = 0.5

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


def run_epoch(loader, training, max_batches=None):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_batches = 0
    tp = 0.0
    fp = 0.0
    fn = 0.0

    grad_ctx = torch.enable_grad() if training else torch.no_grad()
    with grad_ctx:
        for batch_idx, (images, targets, _) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, targets)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).float()

            tp += ((preds == 1) & (targets == 1)).sum().item()
            fp += ((preds == 1) & (targets == 0)).sum().item()
            fn += ((preds == 0) & (targets == 1)).sum().item()

            total_loss += loss.item()
            total_batches += 1

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    mean_loss = total_loss / max(total_batches, 1)
    return mean_loss, precision, recall, f1, total_batches


if "val_loader" not in globals() or val_loader is None:
    val_ds = RawMultiLabelDataset(
        dataset_root,
        split="val",
        image_size=image_size,
        include_background=include_background,
        use_cache=True,
        refresh_cache=False,
    )
    from torch.utils.data import DataLoader
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )

best_val_f1 = -1.0
best_model_path = "raw_multilabel_resnet18_best.pth"

print(f"Device: {device} | epochs: {epochs}")
print(f"Train batches limit: {max_train_batches} | Val batches limit: {max_val_batches}")

for epoch in range(1, epochs + 1):
    train_loss, train_p, train_r, train_f1, train_steps = run_epoch(
        train_loader,
        training=True,
        max_batches=max_train_batches,
    )
    val_loss, val_p, val_r, val_f1, val_steps = run_epoch(
        val_loader,
        training=False,
        max_batches=max_val_batches,
    )

    print(
        f"Epoch {epoch}/{epochs} | "
        f"train loss={train_loss:.4f} p={train_p:.3f} r={train_r:.3f} f1={train_f1:.3f} ({train_steps} steps) | "
        f"val loss={val_loss:.4f} p={val_p:.3f} r={val_r:.3f} f1={val_f1:.3f} ({val_steps} steps)"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "threshold": threshold,
                "best_val_f1": best_val_f1,
            },
            best_model_path,
        )

print(f"Best ResNet val F1: {best_val_f1:.3f}")
print(f"Saved: {best_model_path}")

if swin_prev_val_f1 is None:
    print("Swin baseline not found in current kernel variables.")
else:
    print(f"Swin best val F1 (previous run): {float(swin_prev_val_f1):.3f}")
    print(f"ResNet - Swin delta: {best_val_f1 - float(swin_prev_val_f1):+.3f}")

if vit_prev_val_f1 is None:
    print("ViT baseline not found in current kernel variables.")
else:
    print(f"ViT best val F1 (previous run): {float(vit_prev_val_f1):.3f}")
    print(f"ResNet - ViT delta: {best_val_f1 - float(vit_prev_val_f1):+.3f}")